Search  Engine With Tools and Agents

In [19]:
import os 
from dotenv import load_dotenv

import openai
openai.api_key=os.getenv("OPENAI_API_KEY")
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")

import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "Search_Engine_Tool_Agents"

In [20]:
##Arxiv--Research
##Tools creation
from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

In [21]:
##used the inbuilt tool of wikipedia
api_wrapper_wiki=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=250)
wiki=WikipediaQueryRun(api_wrapper=api_wrapper_wiki)
wiki.name

'wikipedia'

In [22]:
##used the inbuilt tool of arxiv
api_wrapper_arxiv=ArxivAPIWrapper(top_k_results=3,doc_content_chars_max=2000)
arxiv=ArxivQueryRun(api_wrapper=api_wrapper_arxiv)
arxiv.name

'arxiv'

In [23]:
tools=[wiki,arxiv]

In [24]:
##custom tools [RAG tools]
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [25]:
loader=WebBaseLoader("https://docs.langchain.com/")
docs=loader.load()
documents=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200).split_documents(docs)
vectordb=FAISS.from_documents(documents,OpenAIEmbeddings())
retriever=vectordb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000022B7C58ED10>, search_kwargs={})

Convert retriever → tool

Because agents can only call tools, not raw retrievers.

In [26]:
from langchain_core.tools import tool

@tool
def langsmith_search(query: str) -> str:
    """Search any information about LangSmith."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [27]:
tools=[wiki,arxiv,langsmith_search]

In [28]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\GENAI_PROJECTS\\Search_Engine_using_Tools_Agents\\venv\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=250)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=2000)),
 StructuredTool(name='langsmith_search', description='Search any information about LangSmith.', args_schema=<class 'langchain_core.utils.pydantic.langsmith_search'>, func=<function langsmith_search at 0x0000022B7ADD3400>)]

In [29]:
##Run all the tools with Agents and LLM models

#tools+llm model-->agent Executor
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=openai.api_key
)

In [30]:
##prompt template 
from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/openai-functions-agent")

In [31]:
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [32]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=prompt.messages[0].prompt.template
)

In [33]:
agent.invoke(
    {"messages": [("user", "What is LangSmith?")]}
)

{'messages': [HumanMessage(content='What is LangSmith?', additional_kwargs={}, response_metadata={}, id='a89ccf6b-2ffe-4feb-a664-35b5684623d4'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 187, 'total_tokens': 203, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_c55974138f', 'id': 'chatcmpl-DIqeR6m43sjewPjqB0QinVAAU4LQU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ce5f6-91b3-74a0-bde5-af74b45910c3-0', tool_calls=[{'name': 'langsmith_search', 'args': {'query': 'LangSmith'}, 'id': 'call_04oSC53c7qC9qmvfHYYyawBF', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 187, 'outp

In [34]:
result = agent.invoke(
    {"messages": [("user", "What is LangSmith?")]}
)
final_answer = result["messages"][-1].content
print(final_answer)

LangSmith is a platform designed to assist AI teams in using live production data for continuous testing and improvement of their AI agents. It provides several key features:

1. **Observability**: Users can track and understand how their agents operate through detailed tracing and trend metrics.
   
2. **Evaluation**: It allows for the testing and scoring of agent behavior using either production data or offline datasets to enhance performance continuously.

3. **Prompt Engineering**: LangSmith supports iterative improvement of prompts, including version control and collaboration features.

4. **Deployment**: Agents can be deployed with a single click, leveraging scalable infrastructure suitable for long-running tasks.

LangSmith emphasizes high standards of data security and privacy, complying with HIPAA, SOC 2 Type 2, and GDPR. It also includes tools like the LangSmith Agent Builder, which allows users to create AI agents without coding, starting from templates and connecting accoun

In [35]:
result = agent.invoke(
    {"messages": [("user", "What is machine learning?")]}
)
final_answer = result["messages"][-1].content
print(final_answer)

Machine learning (ML) is a field of study within artificial intelligence that focuses on the development and analysis of statistical algorithms capable of learning from data. These algorithms can generalize from the data they are trained on to unseen data, enabling them to perform various tasks without being explicitly programmed for each specific task.


In [36]:
result = agent.invoke(
    {"messages": [("user", "What's the paper 1706.0372 about?")]}
)
final_answer = result["messages"][-1].content
print(final_answer)

It seems that I'm unable to retrieve information directly for the paper with the identifier "1706.0372" from arXiv. Would you like me to help you with something else, or do you want to provide more details about the paper?
